# Belvedere Glacier Open Data — Usage Example

This notebook shows how to discover, filter, and use the Belvedere Glacier monitoring dataset through its **STAC catalog**.

---

## What is a STAC catalog?

**STAC** (SpatioTemporal Asset Catalog) is an open standard for describing geospatial datasets. It organises data into three nested levels:

```
Catalog  (belvedere-glacier)
 └── Collection  (belvedere-monitoring)
      ├── Item  (belv_1977_histo-aerial)  ← one survey epoch
      │    ├── Asset: dsm          → COG .tif
      │    ├── Asset: orthophoto   → COG .tif
      │    └── Asset: pointcloud   → COPC .laz
      ├── Item  (belv_2022_uav)
      └── ...
```

Each **Item** is a GeoJSON Feature: it has a geometry (footprint), a datetime, and a dictionary of **Assets** (the actual files). Because the files are Cloud-Optimized GeoTIFFs (COG) and COPC point clouds, you can stream any spatial or temporal subset directly from Zenodo **without downloading full files**.

### Key benefits
| Benefit | How it works here |
|---|---|
| **Discoverable** | One `catalog.json` entry point indexes all 14 surveys |
| **Filterable** | Query by year, platform, bbox, accuracy |
| **Cloud-native** | COG + COPC allow range-request streaming |
| **Interoperable** | Works with QGIS, Python (`pystac`, `rasterio`), STAC APIs |
| **Self-describing** | Each item carries sensor metadata, CRS, RMSE |

---

## Setup

Install the required packages (already in the `pixi` environment):
```
pixi add pystac rasterio matplotlib
```

In [1]:
import pystac
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

---
## 1. Open the catalog

The catalog can be opened from the **local path** (after running `build_stack.py`) or directly from the **Zenodo URL** once it is published there.

In [2]:
# --- Option A: remote catalog from Zenodo  ---
catalog = pystac.Catalog.from_file(
    "https://franioli.github.io/belvedere-open-data/catalog.json"
)
collection = catalog.get_child("belvedere-monitoring")

# --- Option A: local catalog (clone the repository locally first) ---
# CATALOG_PATH = "./stac_catalog/catalog.json"
# catalog = pystac.Catalog.from_file(CATALOG_PATH)

print(f"Catalog : {catalog.id}")
print(f"Description: {catalog.description}")

Catalog : belvedere-glacier
Description: Belvedere Glacier long-term monitoring Open Data


---
## 2. Browse the collection and list all surveys

In [3]:
collection = catalog.get_child("belvedere-monitoring")
print(f"Collection : {collection.id}")
print(f"Description: {collection.description}")
print(f"Temporal   : {collection.extent.temporal.intervals}")
print()

items = list(collection.get_items())
print(f"{'ID':<30}  {'Date':<12}  {'Platform':<15}  {'RMSE (m)':>8}  {'Assets'}")
print("-" * 90)
for item in sorted(items, key=lambda x: x.datetime):
    date = item.datetime.strftime("%Y-%m-%d")
    platform = item.properties.get("platform", "?")
    rmse = item.properties.get("rmse_global_m", "—")
    asset_list = ", ".join(item.assets.keys())
    print(f"{item.id:<30}  {date:<12}  {platform:<15}  {str(rmse):>8}  {asset_list}")

Collection : belvedere-monitoring
Description: UAV and historical photogrammetric surveys of Belvedere Glacier (1977–2023). Each item contains a DSM (Cloud-Optimized GeoTIFF), an orthophoto (COG), and a point cloud (COPC .laz).
Temporal   : [[datetime.datetime(1977, 9, 16, 0, 0, tzinfo=tzutc()), datetime.datetime(2024, 7, 24, 0, 0, tzinfo=tzutc())]]

ID                              Date          Platform         RMSE (m)  Assets
------------------------------------------------------------------------------------------
belv_1977_histo-aerial          1977-09-16    histo-aerial         0.66  dsm, orthophoto, pointcloud
belv_1991_histo-aerial          1991-08-01    histo-aerial         0.74  dsm, orthophoto, pointcloud
belv_2001_histo-aerial          2001-09-06    histo-aerial         0.38  dsm, orthophoto, pointcloud
belv_2009_digital-aerial        2009-08-01    digital-aerial       0.39  dsm, orthophoto, pointcloud
belv_2015_uav                   2015-10-08    uav                 0.114 

---
## 3. Filter surveys

### 3a. Filter by platform type

In [4]:
uav_items = [i for i in items if i.properties.get("platform") == "uav"]
print(f"UAV surveys ({len(uav_items)}):")
for i in sorted(uav_items, key=lambda x: x.datetime):
    print(f"  {i.id}  —  {i.datetime.year}")

UAV surveys (10):
  belv_2015_uav  —  2015
  belv_2016_uav  —  2016
  belv_2017_uav  —  2017
  belv_2018_uav  —  2018
  belv_2019_uav  —  2019
  belv_2020_uav  —  2020
  belv_2021_uav  —  2021
  belv_2022_uav  —  2022
  belv_2023_uav  —  2023
  belv_2024_uav  —  2024


### 3b. Filter by year range

In [5]:
recent = [i for i in items if i.datetime.year >= 2019]
print(f"Surveys from 2019 onward ({len(recent)}):")
for i in sorted(recent, key=lambda x: x.datetime):
    print(f"  {i.id}")

Surveys from 2019 onward (6):
  belv_2019_uav
  belv_2020_uav
  belv_2021_uav
  belv_2022_uav
  belv_2023_uav
  belv_2024_uav


---
## 4. Inspect an item's assets and get file URLs

In [6]:
item_2022 = collection.get_item("belv_2022_uav")

print(f"Item: {item_2022.id}")
print(f"Date: {item_2022.datetime.strftime('%d %b %Y')}")
print(f"BBox (WGS84): {[round(c, 4) for c in item_2022.bbox]}")
print()
print("Assets:")
for key, asset in item_2022.assets.items():
    print(f"  [{key}]")
    print(f"    URL   : {asset.href}")
    print(f"    Type  : {asset.media_type}")
    print(f"    Roles : {asset.roles}")

Item: belv_2022_uav
Date: 25 Jul 2022
BBox (WGS84): [7.9014, 45.9401, 7.9275, 45.9711]

Assets:
  [dsm]
    URL   : https://zenodo.org/records/10817029/files/belv_2022_uav_dsm_20cm_ortho_cog.tif
    Type  : image/tiff; application=geotiff; profile=cloud-optimized
    Roles : ['data', 'elevation']
  [orthophoto]
    URL   : https://zenodo.org/records/10817029/files/belv_2022_uav_orthophoto_20cm_cog.tif
    Type  : image/tiff; application=geotiff; profile=cloud-optimized
    Roles : ['data', 'visual']
  [pointcloud]
    URL   : https://zenodo.org/records/10817029/files/belv_2022_uav_pcd_copc.laz
    Type  : application/vnd.laszip+copc
    Roles : ['data', 'pointcloud']


---
## 5. Open and visualise a DSM (Cloud-Optimized GeoTIFF)

COG files support **HTTP range requests**: rasterio can read any window or overview level without downloading the full file. Here we read a coarse overview for a quick plot.

In [7]:
dsm_url = item_2022.assets["dsm"].href
print(f"Streaming DSM from:\n  {dsm_url}\n")

with rasterio.open(dsm_url) as src:
    print(f"CRS        : {src.crs}")
    print(f"Resolution : {src.res[0]} m")
    print(f"Size       : {src.width} x {src.height} px")
    print(f"Overviews  : {src.overviews(1)}")

    # Read at a coarse overview (level 4 ≈ 3.2 m/px) for a fast preview
    overview_level = 4
    data = src.read(
        1,
        out_shape=(
            src.height // (2**overview_level),
            src.width // (2**overview_level),
        ),
    )
    nodata = src.nodata

data = data.astype(float)
if nodata is not None:
    data[data == nodata] = np.nan

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(
    data,
    cmap="terrain",
    vmin=np.nanpercentile(data, 2),
    vmax=np.nanpercentile(data, 98),
)
plt.colorbar(im, ax=ax, label="Elevation (m a.s.l.)", fraction=0.03, pad=0.04)
ax.set_title("Belvedere Glacier — DSM 2022 (UAV)", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

Streaming DSM from:
  https://zenodo.org/records/10817029/files/belv_2022_uav_dsm_20cm_ortho_cog.tif



RasterioIOError: HTTP response code: 404

---
## 6. Multi-temporal analysis: DSM of Difference (DoD)

Compare two DSMs to quantify glacier surface elevation change. This reads overview-level data for speed; for full-resolution analysis remove the `out_shape` argument.

In [ ]:
from rasterio.enums import Resampling

YEAR_NEWER = 2022
YEAR_OLDER = 2015
OVERVIEW = 4  # read at 1/16 resolution for speed


def read_dsm_overview(item, overview):
    """Read DSM at a coarse overview level, return (data_array, profile)."""
    url = item.assets["dsm"].href
    with rasterio.open(url) as src:
        h = src.height // (2**overview)
        w = src.width // (2**overview)
        data = src.read(
            1,
            out_shape=(h, w),
            resampling=Resampling.bilinear,
        ).astype(float)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan
        profile = src.profile
    return data, profile


item_newer = collection.get_item(f"belv_{YEAR_NEWER}_uav")
item_older = collection.get_item(f"belv_{YEAR_OLDER}_uav")

print(f"Reading DSM {YEAR_NEWER} …")
dsm_newer, _ = read_dsm_overview(item_newer, OVERVIEW)
print(f"Reading DSM {YEAR_OLDER} …")
dsm_older, _ = read_dsm_overview(item_older, OVERVIEW)
print("Done.")

In [ ]:
# Crop to the overlapping region (both surveys cover the same glacier,
# but their pixel grids may differ slightly at overview resolution).
min_h = min(dsm_newer.shape[0], dsm_older.shape[0])
min_w = min(dsm_newer.shape[1], dsm_older.shape[1])
dod = dsm_newer[:min_h, :min_w] - dsm_older[:min_h, :min_w]

# Symmetric colour scale clipped at ±30 m
clim = 30
cmap = plt.cm.RdBu

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, data, title in zip(
    axes,
    [dsm_older[:min_h, :min_w], dsm_newer[:min_h, :min_w], dod],
    [f"DSM {YEAR_OLDER}", f"DSM {YEAR_NEWER}", f"DoD ({YEAR_NEWER}−{YEAR_OLDER})"],
):
    if "DoD" in title:
        im = ax.imshow(data, cmap=cmap, vmin=-clim, vmax=clim)
        plt.colorbar(im, ax=ax, label="Δ Elevation (m)", fraction=0.046, pad=0.04)
    else:
        im = ax.imshow(
            data,
            cmap="terrain",
            vmin=np.nanpercentile(data, 2),
            vmax=np.nanpercentile(data, 98),
        )
        plt.colorbar(im, ax=ax, label="Elevation (m a.s.l.)", fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

plt.suptitle("Belvedere Glacier — Surface Elevation Change", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Quick statistics on the DoD
valid = dod[np.isfinite(dod)]
print(f"\nDoD statistics ({YEAR_NEWER} vs {YEAR_OLDER}):")
print(f"  Mean  Δh : {np.nanmean(valid):+.2f} m")
print(f"  Median Δh: {np.nanmedian(valid):+.2f} m")
print(f"  Std   Δh : {np.nanstd(valid):.2f} m")

---
## 7. Timeline: survey overview across all epochs

In [ ]:
sorted_items = sorted(items, key=lambda x: x.datetime)
years = [i.datetime.year for i in sorted_items]
rmse = [i.properties.get("rmse_global_m") for i in sorted_items]
platform = [i.properties.get("platform", "unknown") for i in sorted_items]

color_map = {
    "uav": "#2196F3",
    "histo-aerial": "#FF9800",
    "digital-aerial": "#4CAF50",
    "unknown": "#9E9E9E",
}
colors = [color_map.get(p, "#9E9E9E") for p in platform]

fig, ax = plt.subplots(figsize=(13, 4))

for i, (y, r, c, p) in enumerate(zip(years, rmse, colors, platform)):
    ax.bar(y, r if r is not None else 0, color=c, width=0.6, alpha=0.85, label=p)
    if r is not None:
        ax.text(y, r + 0.005, f"{r:.2f}", ha="center", va="bottom", fontsize=8)

# Legend (unique entries only)
seen = set()
handles = []
for p, c in color_map.items():
    if p in platform and p not in seen:
        handles.append(plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.85, label=p))
        seen.add(p)
ax.legend(handles=handles, title="Platform", loc="upper right")

ax.set_xlabel("Year")
ax.set_ylabel("Global RMSE (m)")
ax.set_title("Belvedere Glacier — Photogrammetric Accuracy per Survey")
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45, ha="right")
plt.tight_layout()
plt.show()

---
## 8. Download assets (when you need the full file)

For heavy processing you may want to download the full COG or point cloud first.

In [ ]:
import urllib.request


def download_asset(item, asset_key, out_dir="."):
    """Download one asset from the catalog to a local directory."""
    asset = item.assets[asset_key]
    filename = Path(asset.href).name
    dest = Path(out_dir) / filename
    if dest.exists():
        print(f"Already present: {dest}")
        return dest
    print(f"Downloading {filename} …")
    urllib.request.urlretrieve(asset.href, dest)
    print(f"Saved to {dest}")
    return dest


# Example — uncomment to actually download:
# item = collection.get_item("belv_2022_uav")
# dsm_path = download_asset(item, "dsm", out_dir="/tmp")

print("Asset URLs for belv_2022_uav:")
for key, asset in collection.get_item("belv_2022_uav").assets.items():
    print(f"  {key}: {asset.href}")

---
## 9. Advanced: xDEM for glacier-wide volume change

The `xdem` library (included in the pixi environment) works natively with COG DEMs and can compute co-registered, bias-corrected DoDs. This example sketch shows the workflow for a full analysis.

In [ ]:
# Conceptual workflow — runs on full-resolution data (may take a few minutes)
#
# import xdem
#
# ref_url  = collection.get_item("belv_2022_uav").assets["dsm"].href
# tgt_url  = collection.get_item("belv_2015_uav").assets["dsm"].href
#
# ref_dem = xdem.DEM(ref_url)
# tgt_dem = xdem.DEM(tgt_url).reproject(ref_dem)   # reproject to common grid
#
# # Co-register to remove systematic offsets
# nuth_kaab = xdem.coreg.NuthKaab()
# nuth_kaab.fit(ref_dem, tgt_dem)
# tgt_coreg = nuth_kaab.apply(tgt_dem)
#
# dod = ref_dem - tgt_coreg
# print(f"Mean elevation change 2015–2022: {float(dod.data.mean()):.2f} m")

print("xDEM workflow sketch shown above (uncomment to run on full-resolution data).")

---

## Summary

| Task | Code pattern |
|---|---|
| Open catalog | `pystac.Catalog.from_file(path_or_url)` |
| List all surveys | `collection.get_items()` |
| Filter by property | list comprehension on `item.properties` |
| Get file URL | `item.assets["dsm"].href` |
| Stream a COG window | `rasterio.open(url)` with `out_shape` for overviews |
| Full analysis | download asset, then use `xdem` / `rasterio` |

For questions or issues, contact the dataset maintainers via the Zenodo record.